# 02 — GPU retrievers: Two-Tower and Text

Recommended runtime: NVIDIA L4 High-RAM; A100 is optional. Prerequisites: Notebook 00 and the completed Popularity model from Notebook 01, in the same mode. Two-Tower uses learned item-ID/category representations and a history-mean user tower. Targets and histories are strictly pre-cutoff; histories exclude the target date. Uniform sampled items are unobserved proxies, not verified negatives. Text uses a pretrained MiniLM sentence encoder with a recorded immutable model revision. It is not fine-tuned. Colab downloads that text model only when this notebook is deliberately executed.

Run only in your own Colab session. This notebook mounts Drive and runs the selected pipeline stage. `MODE="smoke"` is the default; smoke outputs are diagnostics, not full H&M results. Full execution has not been run by the authoring agent. Existing source CSVs and CLIP artifacts are read-only. Metadata/images are static snapshots, not proven historical availability.

## Native Colab dependencies
Run in a fresh runtime. A resolver conflict is an error to investigate; do not remove the PyTorch constraints or reinstall CUDA packages to hide it.

In [ ]:
import importlib.metadata
import subprocess
import sys
import tempfile

# Use Colab native Python. Never install the repository's uv environment here.
# Pin any installed PyTorch packages so pip cannot replace Colab CUDA builds.
constraints = []
for package in ("torch", "torchvision", "torchaudio"):
    try:
        constraints.append(f"{package}=={importlib.metadata.version(package)}")
    except importlib.metadata.PackageNotFoundError:
        pass
if not any(value.startswith("torch==") for value in constraints):
    raise RuntimeError("Select a Colab GPU runtime with native CUDA PyTorch.")
with tempfile.NamedTemporaryFile(mode="w", suffix=".txt") as handle:
    handle.write("\n".join(constraints))
    handle.flush()
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--constraint",
            handle.name,
            *[
                "duckdb==1.5.6",
                "pyarrow>=18,<27",
                "faiss-cpu>=1.9,<2",
                "sentence-transformers==3.4.1",
                "transformers==4.57.6",
            ],
        ],
        check=True,
    )

## Shared protocol and validation contracts
Small helpers are repeated so this notebook has no repository-module dependency.

In [ ]:
import hashlib
import json
import math
from datetime import date, timedelta

import numpy as np
import pyarrow as pa

PROTOCOL_VERSION = "1.0"
ARTIFACT_VERSION = "hm-mvp-1.0.0"
FEATURE_SCHEMA_VERSION = "ranking-1.0"
SEED = 20261031
CUTOFF = date(2020, 7, 28)
CHANNELS = ("popularity", "itemcf", "als", "two_tower", "text", "image")
RRF_K0 = 60
CANDIDATES_PER_CHANNEL = 100
FUSED_CANDIDATE_BUDGET = 500
CANDIDATE_SCHEMA = pa.schema(
    [
        ("query_id", pa.string()),
        ("customer_id", pa.string()),
        ("anchor_date", pa.date32()),
        ("article_id", pa.string()),
        ("channel", pa.string()),
        ("raw_score", pa.float32()),
        ("source_rank", pa.int32()),
        ("retriever_version", pa.string()),
    ]
)


def require(condition, message):
    if not condition:
        raise ValueError(message)


def identity(value):
    payload = json.dumps(value, sort_keys=True, separators=(",", ":"), default=str)
    return hashlib.sha256(payload.encode()).hexdigest()


def anchor_schedule():
    starts = [
        ("ranking_train", CUTOFF),
        ("validation", date(2020, 8, 16)),
        ("test", date(2020, 9, 4)),
    ]
    return [
        {
            "split": split,
            "anchor_date": start + timedelta(days=offset),
            "window_end": start + timedelta(days=offset + 7),
        }
        for split, start in starts
        for offset in (0, 3, 6, 9, 12)
    ]


def unique_ranked(items):
    return list(dict.fromkeys(items))


def query_metrics(predictions, truth, k=20):
    truth = set(truth)
    require(bool(truth), "Empty ground truth is not an eligible metric denominator.")
    predictions = unique_ranked(predictions)[:k]
    hits = [int(item in truth) for item in predictions]
    recall = sum(hits) / len(truth)
    dcg = sum(hit / math.log2(j + 2) for j, hit in enumerate(hits))
    ideal = sum(1 / math.log2(j + 2) for j in range(min(k, len(truth))))
    cumulative, precision_sum = 0, 0.0
    for j, hit in enumerate(hits, 1):
        cumulative += hit
        precision_sum += hit * cumulative / j
    return {
        "recall": recall,
        "ndcg": dcg / ideal,
        "ap": precision_sum / min(k, len(truth)),
    }


def stable_top(items, scores, limit=100):
    best = {}
    for item, score in zip(items, scores, strict=True):
        score = float(score)
        require(math.isfinite(score), "Nonfinite retrieval score.")
        best[item] = max(best.get(item, -math.inf), score)
    return sorted(best.items(), key=lambda pair: (-pair[1], pair[0]))[:limit]


def fuse_query(rows, excluded=None, budget=500):
    by_item = {}
    keys = set()
    for row in rows:
        if row["channel"] == excluded:
            continue
        key = row["channel"], row["article_id"]
        require(key not in keys, "Duplicate source candidate before fusion.")
        keys.add(key)
        entry = by_item.setdefault(
            row["article_id"],
            {"article_id": row["article_id"], "rrf": 0.0, "sources": {}},
        )
        entry["sources"][row["channel"]] = (row["source_rank"], row["raw_score"])
        entry["rrf"] += 1.0 / (RRF_K0 + row["source_rank"])
    ordered = sorted(by_item.values(), key=lambda row: (-row["rrf"], row["article_id"]))
    return ordered[:budget], len(ordered)


def validate_candidates(table, roster, catalog, channel):
    require(table.schema.equals(CANDIDATE_SCHEMA), "Candidate schema mismatch.")
    query_lookup = {r["query_id"]: r for r in roster}
    seen, groups = set(), {}
    for row in table.to_pylist():
        qid, item = row["query_id"], row["article_id"]
        require(qid in query_lookup, "Candidate references an unknown query.")
        q = query_lookup[qid]
        require(
            row["customer_id"] == q["customer_id"]
            and row["anchor_date"] == q["anchor_date"],
            "Query identity mismatch.",
        )
        require(item in catalog and isinstance(item, str), "Ineligible candidate ID.")
        require(row["channel"] == channel, "Wrong channel in candidate shard.")
        require(
            row["retriever_version"] == ARTIFACT_VERSION, "Retriever version mismatch."
        )
        require((qid, item) not in seen, "Duplicate query/article candidate.")
        require(math.isfinite(row["raw_score"]), "Nonfinite candidate score.")
        seen.add((qid, item))
        groups.setdefault(qid, []).append(row)
    for rows in groups.values():
        rows.sort(key=lambda r: r["source_rank"])
        require(len(rows) <= 100, "More than 100 candidates for a channel.")
        require(
            [r["source_rank"] for r in rows] == list(range(1, len(rows) + 1)),
            "Source ranks are not contiguous from one.",
        )
        expected = stable_top(
            [r["article_id"] for r in rows], [r["raw_score"] for r in rows]
        )
        require(
            [r["article_id"] for r in rows] == [i for i, _ in expected],
            "Unstable score/article ordering.",
        )
    require(
        set(groups) == set(query_lookup), "A query silently disappeared from a channel."
    )


def validate_mapping(ids, rows):
    require(len(ids) == len(rows), "Mapping length mismatch.")
    require(len(ids) == len(set(ids)), "Duplicate article ID in mapping.")
    require(list(rows) == list(range(len(ids))), "Noncontiguous embedding rows.")
    require(
        all(isinstance(x, str) and len(x) == 10 and x.isdigit() for x in ids),
        "Article IDs must preserve ten-digit strings.",
    )


def validate_manifest(manifest, mode, catalog_id=None):
    fields = {
        "artifact_version",
        "protocol_version",
        "mode",
        "input_paths",
        "input_fingerprints",
        "model_parameters",
        "random_seed",
        "row_count",
        "snapshot_or_anchor",
        "catalog_id",
        "feature_schema_version",
        "output_paths",
        "validation_status",
        "files",
        "stage",
    }
    require(fields <= manifest.keys(), "Incomplete artifact manifest.")
    from pathlib import PurePosixPath

    require(
        set(manifest["files"]) == set(manifest["output_paths"]),
        "Output manifest mismatch.",
    )
    for name in manifest["files"]:
        path = PurePosixPath(name)
        require(
            not path.is_absolute() and ".." not in path.parts and "\\" not in name,
            "Unsafe artifact-relative path.",
        )
    require(
        manifest["artifact_version"] == ARTIFACT_VERSION
        and manifest["protocol_version"] == PROTOCOL_VERSION,
        "Artifact/protocol mismatch.",
    )
    require(manifest["mode"] == mode, "Smoke/full artifact mixing is forbidden.")
    require(manifest["validation_status"] == "validated", "Artifact is not complete.")
    if catalog_id is not None:
        require(manifest["catalog_id"] == catalog_id, "Catalog identity mismatch.")


def metric_smoke_checks():
    result = query_metrics(["a", "x", "b"], {"a", "b"}, 3)
    require(result["recall"] == 1, "Recall formula failed.")
    require(abs(result["ap"] - (1 + 2 / 3) / 2) < 1e-12, "AP formula failed.")
    require(query_metrics([], {"a"})["ndcg"] == 0, "Missing recommendations lost.")
    require(
        query_metrics(["a", "a"], {"a", "b"}, 2)["recall"] == 0.5,
        "Duplicate recommendation received extra credit.",
    )
    schedule = anchor_schedule()
    require(len(schedule) == 15, "Anchor count mismatch.")
    require(schedule[-1]["window_end"] == date(2020, 9, 23), "Test overflow.")

## Runtime, storage and one execution mode
Change only `MODE` between `smoke` and `full`. Outputs use `<stage>/mvp_v1/<mode>/` beneath the specified Drive data root. Completion manifests are written last. Changed inputs or parameters require a new artifact version; completed runs are immutable. Local disk checks include a reserve, but cannot guarantee Colab or Drive quota.

In [ ]:
import os
import shutil
import time
import uuid
from pathlib import Path

import duckdb
import pyarrow.parquet as pq

DRIVE_ROOT = Path("/content/drive/MyDrive/projects/e_commerce_recsys/data")
LOCAL_ROOT = Path("/content/data/hm")
WORK_ROOT = Path("/content/recsys_work")
MODE = "smoke"  # 'smoke' or 'full'; smoke results are diagnostics only.
SMOKE_CUSTOMERS = 2000
SMOKE_QUERIES_PER_ANCHOR = 32
MAX_HISTORY_ITEMS = 100  # Most recent distinct items; query representation only.
RUN_NAMESPACE = "mvp_v1"


def resource_policy(mode, cpus, usable_bytes):
    require(mode in ("smoke", "full"), "MODE must be smoke or full.")
    gib = usable_bytes / 1024**3
    workers = max(1, min(int(cpus), 2 if mode == "smoke" else 8))
    memory = max(0.25, min(4 if mode == "smoke" else 16, gib / 4))
    batch = 256
    if mode == "full":
        batch = 4096 if gib >= 64 else 2048 if gib >= 16 else 512 if gib >= 8 else 256
    return {
        "threads": workers,
        "memory_limit": f"{memory:.2f}GB",
        "query_batch_size": batch,
        "target_shard_bytes": (8 if mode == "smoke" else 192) * 1024**2,
        "usable_ram_bytes": int(usable_bytes),
        "available_cpus": int(cpus),
    }


def detect_resources(proc=Path("/proc"), cgroup=Path("/sys/fs/cgroup")):
    cpu_limits = [os.cpu_count() or 1]
    if hasattr(os, "sched_getaffinity"):
        cpu_limits.append(len(os.sched_getaffinity(0)))
    memory_limits = []
    meminfo = proc / "meminfo"
    if meminfo.is_file():
        values = {
            line.split(":")[0]: int(line.split()[1]) * 1024
            for line in meminfo.read_text().splitlines()
            if line.startswith(("MemAvailable:", "MemTotal:"))
        }
        memory_limits.append(
            values.get("MemAvailable", values.get("MemTotal", 4 * 1024**3))
        )
    directories = {cgroup, cgroup / "memory", cgroup / "cpu", cgroup / "cpu,cpuacct"}
    membership = proc / "self/cgroup"
    if membership.is_file():
        for line in membership.read_text().splitlines():
            _, controllers, relative = line.split(":", 2)
            roots = [cgroup] if not controllers else [cgroup / controllers]
            roots += [cgroup / name for name in controllers.split(",") if name]
            for root in roots:
                child = root / relative.lstrip("/")
                # Include ancestor quotas; a child may say 'max' under a finite parent.
                while child == root or root in child.parents:
                    directories.add(child)
                    if child == root:
                        break
                    child = child.parent
    for directory in directories:
        for limit_name, used_name in [
            ("memory.max", "memory.current"),
            ("memory.limit_in_bytes", "memory.usage_in_bytes"),
        ]:
            limit_path, used_path = directory / limit_name, directory / used_name
            if limit_path.is_file():
                text = limit_path.read_text().strip()
                if text.isdigit() and int(text) < 1 << 60:
                    used = int(used_path.read_text()) if used_path.is_file() else 0
                    memory_limits.append(max(0, int(text) - used))
        quota = directory / "cpu.max"
        if quota.is_file():
            amount, period = quota.read_text().split()
            if amount != "max":
                cpu_limits.append(max(1, int(amount) // int(period)))
        quota, period = directory / "cpu.cfs_quota_us", directory / "cpu.cfs_period_us"
        if quota.is_file() and period.is_file() and int(quota.read_text()) > 0:
            cpu_limits.append(max(1, int(quota.read_text()) // int(period.read_text())))
    # Unknown memory is deliberately conservative; no high-RAM assumption on other OSes.
    return min(cpu_limits), min(memory_limits) if memory_limits else 4 * 1024**3


def configure_resources():
    global THREADS, MEMORY_LIMIT, QUERY_BATCH_SIZE, GPU_RESOURCES
    cpus, memory = detect_resources()
    GPU_RESOURCES = resource_policy(MODE, cpus, memory)
    THREADS = GPU_RESOURCES["threads"]
    MEMORY_LIMIT = GPU_RESOURCES["memory_limit"]
    QUERY_BATCH_SIZE = GPU_RESOURCES["query_batch_size"]
    return GPU_RESOURCES


def rss_gib():
    import resource
    import sys

    peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return peak / (1024**3 if sys.platform == "darwin" else 1024**2)


def performance_event(operation, started, **fields):
    record = {
        "operation": operation,
        "seconds": time.perf_counter() - started,
        "process_peak_RSS_GiB": rss_gib(),
        **gpu_memory_stats(),
        **fields,
    }
    GPU_PERFORMANCE.append(record)
    print(json.dumps(record, sort_keys=True))
    return record


GPU_PERFORMANCE = []
GPU_RESOURCES = resource_policy(MODE, *detect_resources())
THREADS = GPU_RESOURCES["threads"]
MEMORY_LIMIT = GPU_RESOURCES["memory_limit"]
QUERY_BATCH_SIZE = GPU_RESOURCES["query_batch_size"]


def sql_string(value):
    return "'" + str(value).replace("'", "''") + "'"


def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024**2), b""):
            h.update(chunk)
    return h.hexdigest()


def disk_guard(path, needed_bytes, operation):
    root = Path(path)
    while not root.exists():
        root = root.parent
    free = shutil.disk_usage(root).free
    require(
        free >= needed_bytes + 2 * 1024**3,
        f"{operation}: need {needed_bytes / 1024**3:.2f} GiB plus 2 GiB reserve; "
        f"only {free / 1024**3:.2f} GiB free at {root}. No population truncated.",
    )


def copy_verified(source, target, expected=None):
    source, target = Path(source), Path(target)
    require(source.is_file(), f"Missing prerequisite file: {source}")
    digest = expected or sha256_file(source)
    if target.is_file() and target.stat().st_size == source.stat().st_size:
        if sha256_file(target) == digest:
            return target
    disk_guard(target.parent, source.stat().st_size, "Stage input")
    target.parent.mkdir(parents=True, exist_ok=True)
    partial = target.with_suffix(target.suffix + ".partial")
    shutil.copyfile(source, partial)
    require(sha256_file(partial) == digest, "Staged file checksum mismatch.")
    os.replace(partial, target)
    return target


def stage_path(stage, root):
    return Path(root) / stage / RUN_NAMESPACE / MODE


def load_stage(stage, catalog_id=None, wanted=None):
    remote = stage_path(stage, DRIVE_ROOT)
    marker = remote / "_COMPLETE.json"
    require(marker.is_file(), f"Complete prerequisite missing: {marker}")
    manifest = json.loads(marker.read_text())
    validate_manifest(manifest, MODE, catalog_id)
    require(manifest["stage"] == stage, "Wrong stage manifest.")
    local = stage_path(stage, WORK_ROOT)
    if local.exists():
        unexpected = [
            p
            for p in local.rglob("*")
            if p.is_file()
            and p.relative_to(local).as_posix() not in manifest["files"]
            and p.name != "_COMPLETE.json"
        ]
        if unexpected:
            local.rename(local.with_name(f"{MODE}.incomplete-{time.time_ns()}"))
    local.mkdir(parents=True, exist_ok=True)
    names = manifest["files"] if wanted is None else wanted
    for name in names:
        require(name in manifest["files"], f"Artifact file not declared: {name}")
        info = manifest["files"][name]
        require(
            (remote / name).stat().st_size == info["bytes"], "Published size mismatch."
        )
        require(
            sha256_file(remote / name) == info["sha256"],
            "Published source checksum mismatch.",
        )
        copy_verified(remote / name, local / name, info["sha256"])
    return local, manifest


def reusable(stage, parameters, inputs, catalog_id):
    marker = stage_path(stage, DRIVE_ROOT) / "_COMPLETE.json"
    if not marker.exists():
        return None
    path, manifest = load_stage(stage, catalog_id)
    require(
        manifest["model_parameters"] == parameters,
        "Existing parameters differ; version the run.",
    )
    require(
        manifest["input_fingerprints"] == inputs,
        "Existing inputs differ; version the run.",
    )
    return path, manifest


def runtime_versions():
    from importlib.metadata import PackageNotFoundError, version

    versions = {}
    for name in (
        "duckdb",
        "pyarrow",
        "numpy",
        "scipy",
        "implicit",
        "faiss-cpu",
        "torch",
        "transformers",
        "sentence-transformers",
        "scikit-learn",
        "lightgbm",
        "joblib",
    ):
        try:
            versions[name] = version(name)
        except PackageNotFoundError:
            continue
    return versions


def publish_copy(source, target, expected=None):
    source, target = Path(source), Path(target)
    digest = expected or sha256_file(source)
    if target.exists():
        require(
            target.is_file() and sha256_file(target) == digest,
            f"Conflicting incomplete artifact retained at {target}; "
            "manual review required.",
        )
        return target
    target.parent.mkdir(parents=True, exist_ok=True)
    partial = target.with_name(target.name + "." + uuid.uuid4().hex + ".partial")
    shutil.copyfile(source, partial)
    require(sha256_file(partial) == digest, "Publication checksum mismatch.")
    require(not target.exists(), "Concurrent publication detected; retry verification.")
    # Never rename or delete an existing remote artifact.
    os.rename(partial, target)
    return target


def publish(stage, local, parameters, inputs, catalog_id, row_count, snapshot=None):
    local = Path(local)
    files = {}
    for path in sorted(local.rglob("*")):
        if (
            path.is_file()
            and path.name != "_COMPLETE.json"
            and not path.name.endswith(".partial")
        ):
            name = path.relative_to(local).as_posix()
            files[name] = {"bytes": path.stat().st_size, "sha256": sha256_file(path)}
    require(bool(files), "Refusing to publish an empty stage.")
    manifest = {
        "stage": stage,
        "artifact_version": ARTIFACT_VERSION,
        "protocol_version": PROTOCOL_VERSION,
        "mode": MODE,
        "input_paths": sorted(inputs),
        "input_fingerprints": inputs,
        "model_parameters": parameters,
        "random_seed": SEED,
        "row_count": int(row_count),
        "snapshot_or_anchor": str(snapshot or CUTOFF),
        "catalog_id": catalog_id,
        "feature_schema_version": FEATURE_SCHEMA_VERSION,
        "output_paths": sorted(files),
        "validation_status": "validated",
        "files": files,
        "runtime_versions": runtime_versions(),
        "execution": {"implementation": "gpu-execution-2", "resources": GPU_RESOURCES},
    }
    remote = stage_path(stage, DRIVE_ROOT)
    existing = remote / "_COMPLETE.json"
    if existing.exists():
        require(
            json.loads(existing.read_text()) == manifest,
            "Refusing to replace a completed stage.",
        )
        return manifest
    disk_guard(
        remote, sum(x["bytes"] for x in files.values()), "Publish validated stage"
    )
    remote.mkdir(parents=True, exist_ok=True)
    for name, info in files.items():
        publish_copy(local / name, remote / name, info["sha256"])
    # Completion marker is published last, after every checksum is verified.
    marker = local / "_COMPLETE.json"
    marker.write_text(json.dumps(manifest, indent=2, sort_keys=True))
    publish_copy(marker, existing)
    return manifest


def new_stage(stage):
    path = stage_path(stage, WORK_ROOT)
    if path.exists() and any(path.iterdir()):
        path.rename(path.with_name(f"{MODE}.incomplete-{time.time_ns()}"))
    path.mkdir(parents=True, exist_ok=True)
    # Completion manifests govern reuse of incomplete local runs.
    return path


def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, sort_keys=True, default=str))


def connect_db():
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    db = duckdb.connect()
    db.execute(f"SET memory_limit={sql_string(MEMORY_LIMIT)}")
    db.execute(f"SET threads={THREADS}")
    db.execute(f"SET temp_directory={sql_string(WORK_ROOT / 'spill')}")
    db.execute("SET preserve_insertion_order=false")
    return db


def export_query(db, query, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    partial = path.with_suffix(".partial")
    db.execute(
        f"COPY ({query}) TO {sql_string(partial)} "
        "(FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 65536)"
    )
    os.replace(partial, path)


def view_file(db, name, path):
    db.execute(
        f"CREATE OR REPLACE VIEW {name} AS "
        f"SELECT * FROM read_parquet({sql_string(path)})"
    )


def query_batches(db, anchor):
    previous = ""
    while True:
        rows = db.execute(
            (
                "SELECT query_id, customer_id, anchor_date, split FROM "
                "queries WHERE anchor_date=? AND eligible AND query_id>? "
                "ORDER BY query_id LIMIT ?"
            ),
            [anchor, previous, QUERY_BATCH_SIZE],
        ).fetchall()
        if not rows:
            return
        batch = [
            dict(
                zip(
                    ("query_id", "customer_id", "anchor_date", "split"),
                    row,
                    strict=True,
                )
            )
            for row in rows
        ]
        yield batch
        previous = batch[-1]["query_id"]


def attach_queries(db, batch):
    db.register("batch_queries", pa.Table.from_pylist(batch))


def history_for_batch(db, batch, anchor):
    attach_queries(db, batch)
    # Exact historical counts; only the query preference representation is capped.
    rows = db.execute(
        (
            "WITH grouped AS (\n"
            "        SELECT t.customer_id, t.article_id, count(*) AS n, "
            "max(t.t_dat) AS last_date\n"
            "        FROM transactions t JOIN batch_queries b "
            "USING(customer_id)\n"
            "        WHERE t.t_dat < ? GROUP BY "
            "t.customer_id,t.article_id)\n"
            "        SELECT * FROM grouped QUALIFY row_number() OVER\n"
            "        (PARTITION BY customer_id ORDER BY last_date DESC, "
            "article_id) <= ?\n"
            "        ORDER BY customer_id,last_date DESC,article_id"
        ),
        [anchor, MAX_HISTORY_ITEMS],
    ).fetchall()
    result = {}
    for customer, item, count, last in rows:
        weight = float(count) * math.exp(-(anchor - last).days / 30.0)
        result.setdefault(customer, []).append((item, weight))
    return result


def open_context():
    processed, pm = load_stage(
        "processed",
        wanted=[
            "transactions.parquet",
            "articles.parquet",
            "user_id_map.parquet",
            "article_id_map.parquet",
        ],
    )
    protocol_marker = stage_path("protocol", DRIVE_ROOT) / "_COMPLETE.json"
    protocol_manifest = json.loads(protocol_marker.read_text())
    wanted = [
        name
        for name in protocol_manifest["files"]
        if name in ("eligible_catalog.parquet", "anchors.parquet")
        or name.startswith("queries/")
    ]
    protocol, qm = load_stage("protocol", pm["catalog_id"], wanted=wanted)
    db = connect_db()
    for name in ("transactions", "articles", "user_id_map", "article_id_map"):
        view_file(db, name, processed / f"{name}.parquet")
    for name in ("eligible_catalog", "anchors"):
        view_file(db, name, protocol / f"{name}.parquet")
    view_file(db, "queries", protocol / "queries" / "*" / "queries.parquet")
    catalog = [
        r[0]
        for r in db.execute(
            "SELECT article_id FROM eligible_catalog ORDER BY article_id"
        ).fetchall()
    ]
    return db, catalog, pm, qm


def runtime_start():
    require(MODE in ("smoke", "full"), "MODE must be smoke or full.")
    from google.colab import drive

    drive.mount("/content/drive")
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
    np.random.seed(SEED)
    settings = configure_resources()
    print(
        {
            "mode": MODE,
            "resources": settings,
            "result_scope": "diagnostic" if MODE == "smoke" else "full protocol",
            "local_free_GiB": round(shutil.disk_usage(WORK_ROOT).free / 1024**3, 2),
        }
    )


def gpu_memory_stats():
    import sys

    torch = sys.modules.get("torch")
    if torch is None or not torch.cuda.is_initialized():
        return {}
    return {
        "gpu_peak_allocated_GiB": torch.cuda.max_memory_allocated() / 1024**3,
        "gpu_peak_reserved_GiB": torch.cuda.max_memory_reserved() / 1024**3,
    }

## Shared bounded retrieval and candidate contract

In [ ]:
import pyarrow.compute as pc


def training_graph(db, catalog):
    from scipy import sparse

    users = [
        r[0]
        for r in db.execute(
            "SELECT customer_id FROM user_id_map ORDER BY user_idx"
        ).fetchall()
    ]
    db.register(
        "model_items",
        pa.table({"article_id": catalog, "model_row": range(len(catalog))}),
    )
    edges = db.execute(
        "SELECT u.user_idx AS u,i.model_row AS i,count(*) AS n\n"
        "        FROM transactions t JOIN user_id_map u "
        "USING(customer_id)\n"
        "        JOIN model_items i USING(article_id) WHERE t.t_dat <"
        " DATE '2020-07-28'\n"
        "        GROUP BY u.user_idx,i.model_row ORDER BY "
        "u.user_idx,i.model_row"
    )
    # Stream typed arrays; a complete sparse graph, never a dense user-item matrix.
    u_chunks, i_chunks, n_chunks = [], [], []
    for batch in edges.to_arrow_reader(65536):
        u_chunks.append(batch.column(0).to_numpy().astype(np.int32))
        i_chunks.append(batch.column(1).to_numpy().astype(np.int32))
        n_chunks.append(batch.column(2).to_numpy().astype(np.float32))
    require(bool(u_chunks), "Empty retrieval-training graph.")
    graph = sparse.csr_matrix(
        (
            np.concatenate(n_chunks),
            (np.concatenate(u_chunks), np.concatenate(i_chunks)),
        ),
        shape=(len(users), len(catalog)),
        dtype=np.float32,
    )
    graph.sort_indices()
    return graph, users


def itemcf_neighbors(graph, top_k=64, block_size=32):
    from scipy import sparse

    binary = graph.copy()
    binary.data[:] = 1.0
    degree = np.diff(binary.indptr).astype(np.float32)
    # Keep every historical edge; downweight high-degree users instead of dropping them.
    weighted = sparse.diags(1 / np.sqrt(np.maximum(degree, 1))) @ binary
    norms = np.sqrt(np.asarray(weighted.power(2).sum(axis=0)).ravel())
    normalized = (weighted @ sparse.diags(1 / np.maximum(norms, 1e-12))).tocsr()
    blocks = []
    n_items = graph.shape[1]
    for start in range(0, n_items, block_size):
        product = (normalized[:, start : start + block_size].T @ normalized).tocsr()
        row_indices, col_indices, values = [], [], []
        for local_row in range(product.shape[0]):
            a, b = product.indptr[local_row : local_row + 2]
            columns, scores = product.indices[a:b], product.data[a:b]
            keep = (columns != start + local_row) & (scores > 0)
            columns, scores = columns[keep], scores[keep]
            order = np.lexsort((columns, -scores))[:top_k]
            row_indices.extend([local_row] * len(order))
            col_indices.extend(columns[order])
            values.extend(scores[order])
        blocks.append(
            sparse.csr_matrix(
                (values, (row_indices, col_indices)),
                shape=(product.shape[0], n_items),
                dtype=np.float32,
            )
        )
    result = sparse.vstack(blocks, format="csr")
    require(
        np.diff(result.indptr).max(initial=0) <= top_k, "Unbounded ItemCF neighbors."
    )
    return result


def flat_index(vectors):
    import faiss

    index = faiss.IndexFlatIP(vectors.shape[1])
    for start in range(0, len(vectors), 4096):
        index.add(np.ascontiguousarray(vectors[start : start + 4096], dtype=np.float32))
    return index


def index_search(index, queries, ids, limit=100):
    queries = np.ascontiguousarray(queries, dtype=np.float32)
    require(len(ids) == index.ntotal, "FAISS row map length mismatch.")
    if not len(ids):
        return [[] for _ in queries]
    scores, positions = index.search(queries, min(limit + 1, len(ids)))
    output = []
    for q, row_scores, row_positions in zip(queries, scores, positions, strict=True):
        valid = row_positions >= 0
        row_scores, row_positions = row_scores[valid], row_positions[valid]
        # Include every boundary tie before lexicographic ID tie-breaking.
        if len(row_scores) > limit and row_scores[limit] == row_scores[limit - 1]:
            threshold = float(
                np.nextafter(np.float32(row_scores[limit - 1]), np.float32(-np.inf))
            )
            _, row_scores, row_positions = index.range_search(q[None, :], threshold)
        output.append(
            stable_top([ids[int(i)] for i in row_positions], row_scores.tolist(), limit)
        )
    return output


def vector_preferences(batch, histories, vectors, id_to_row):
    result = np.zeros((len(batch), vectors.shape[1]), dtype=np.float32)
    for i, query in enumerate(batch):
        history = histories.get(query["customer_id"], [])
        for item, weight in history:
            row = id_to_row.get(item)
            if row is not None:
                result[i] += vectors[row] * weight
    norms = np.linalg.norm(result, axis=1)
    valid = norms > 1e-12
    result[valid] /= norms[valid, None]
    return result, valid


def candidate_stage(split, channel, anchor):
    return f"candidates/{split}/{channel}/{anchor}"


def generate_channel(
    db, catalog, channel, model_manifest, protocol_manifest, retrieve_batch
):
    catalog_set = set(catalog)
    params = {
        "channel": channel,
        "per_channel": 100,
        "query_batch_size": QUERY_BATCH_SIZE,
        "history_item_limit": MAX_HISTORY_ITEMS,
        "history_decay_days": 30,
    }
    for anchor_info in anchor_schedule():
        anchor, split = anchor_info["anchor_date"], anchor_info["split"]
        stage = candidate_stage(split, channel, anchor)
        inputs = {
            "model_manifest": identity(model_manifest),
            "protocol_manifest": identity(protocol_manifest),
        }
        if reusable(stage, params, inputs, identity(catalog)) is not None:
            continue
        local = new_stage(stage)
        count = db.execute(
            "SELECT count(*) FROM queries WHERE eligible AND anchor_date=?", [anchor]
        ).fetchone()[0]
        disk_guard(local, count * 100 * 180, "Candidate anchor output estimate")
        total, fallback_count, seen_queries = 0, 0, 0
        for batch_number, batch in enumerate(query_batches(db, anchor)):
            histories = history_for_batch(db, batch, anchor)
            results, fallbacks = retrieve_batch(batch, histories)
            require(len(results) == len(batch), "Retriever omitted queries.")
            rows = []
            for query, results_for_query in zip(batch, results, strict=True):
                # Sort after float32 conversion, matching persisted score precision.
                ranked = stable_top(
                    [i for i, _ in results_for_query],
                    [float(np.float32(s)) for _, s in results_for_query],
                )
                for rank, (item, score) in enumerate(ranked, 1):
                    rows.append(
                        {
                            **{
                                k: query[k]
                                for k in ("query_id", "customer_id", "anchor_date")
                            },
                            "article_id": item,
                            "channel": channel,
                            "raw_score": score,
                            "source_rank": rank,
                            "retriever_version": ARTIFACT_VERSION,
                        }
                    )
            table = pa.Table.from_pylist(rows, schema=CANDIDATE_SCHEMA)
            validate_candidates(table, batch, catalog_set, channel)
            pq.write_table(
                table, local / f"shard-{batch_number:05d}.parquet", compression="zstd"
            )
            total += len(rows)
            fallback_count += int(fallbacks)
            seen_queries += len(batch)
        require(seen_queries == count, "Full query population was not processed.")
        write_json(
            local / "diagnostics.json",
            {
                "queries": count,
                "candidate_rows": total,
                "fallback_queries": fallback_count,
            },
        )
        publish(stage, local, params, inputs, identity(catalog), total, anchor)
        print(
            {"channel": channel, "anchor": str(anchor), "queries": count, "rows": total}
        )


def content_retriever(vectors, ids, index, fallback):
    id_to_row = {item: i for i, item in enumerate(ids)}

    def retrieve(batch, histories):
        queries, valid = vector_preferences(batch, histories, vectors, id_to_row)
        results = index_search(index, queries, ids)
        for i in np.flatnonzero(~valid):
            results[int(i)] = fallback
        return results, int((~valid).sum())

    return retrieve


def gpu_candidate_parameters(channel):
    # Legacy compatibility descriptor, NOT the actual optimized execution settings.
    # Reader 03 compares this dict across anchors. Actual settings live in execution.
    result = {
        "channel": channel,
        "per_channel": 100,
        "query_batch_size": 256,
        "history_item_limit": MAX_HISTORY_ITEMS,
        "history_decay_days": 30,
    }
    discovered = None
    for info in anchor_schedule():
        stage = candidate_stage(info["split"], channel, info["anchor_date"])
        path = stage_path(stage, DRIVE_ROOT) / "_COMPLETE.json"
        if not path.exists():
            continue
        manifest = json.loads(path.read_text())
        validate_manifest(manifest, MODE)
        params = manifest["model_parameters"]
        require(
            all(
                params.get(k) == v for k, v in result.items() if k != "query_batch_size"
            ),
            "Completed candidate semantics differ; explicit migration required.",
        )
        require(
            set(params) == set(result)
            and isinstance(params["query_batch_size"], int)
            and params["query_batch_size"] > 0,
            "Unknown candidate compatibility descriptor.",
        )
        if discovered is not None:
            require(
                params == discovered, "Existing anchors have incompatible descriptors."
            )
        discovered = params
    return discovered or result


def candidate_reuse(db, stage, parameters, inputs, catalog_id, anchor, roster):
    path = stage_path(stage, DRIVE_ROOT)
    marker = path / "_COMPLETE.json"
    if not marker.exists():
        return None
    started = time.perf_counter()
    manifest = json.loads(marker.read_text())
    validate_manifest(manifest, MODE, catalog_id)
    require(
        manifest["stage"] == stage and manifest["snapshot_or_anchor"] == str(anchor),
        "Completed candidate stage identity mismatch.",
    )
    require(
        manifest["model_parameters"] == parameters
        and manifest["input_fingerprints"] == inputs,
        "Completed candidates are incompatible; not overwritten.",
    )
    rows = 0
    files = []
    for name, info in manifest["files"].items():
        file = path / name
        require(
            file.is_file() and file.stat().st_size == info["bytes"],
            "Published size mismatch.",
        )
        require(sha256_file(file) == info["sha256"], "Published checksum mismatch.")
        if name.startswith("shard-") and name.endswith(".parquet"):
            parquet = pq.ParquetFile(file)
            require(
                parquet.schema_arrow.remove_metadata().equals(CANDIDATE_SCHEMA),
                "Completed candidate schema mismatch.",
            )
            rows += parquet.metadata.num_rows
            files.append(file)
    require(files and rows == manifest["row_count"], "Completed row totals mismatch.")
    # Prior validation_status plus checked bytes preserve the original validation proof.
    require(
        len(roster) <= rows <= len(roster) * 100,
        "Completed query/row counts impossible.",
    )
    if parameters["channel"] in ("popularity", "als"):
        require(
            rows
            == len(roster)
            * min(
                100, db.execute("SELECT count(*) FROM eligible_catalog").fetchone()[0]
            ),
            "Completed fixed-length candidate count mismatch.",
        )
    performance_event(
        "verified_completed_anchor",
        started,
        stage=stage,
        rows=rows,
        files=len(files),
        copied_candidate_files=0,
    )
    return manifest


def anchor_roster(db, anchor):
    table = db.execute(
        """SELECT query_id,customer_id,anchor_date FROM queries
        WHERE eligible AND anchor_date=? ORDER BY query_id""",
        [anchor],
    ).to_arrow_table()
    require(len(table) > 0, "No eligible queries; no population silently skipped.")
    require(len(pc.unique(table["query_id"])) == len(table), "Duplicate query IDs.")
    require(
        len(pc.unique(table["customer_id"])) == len(table),
        "Repeated customer within anchor.",
    )
    return table


def prepare_anchor_history(db, roster, anchor):
    started = time.perf_counter()
    db.register(
        "gpu_anchor_queries",
        roster.append_column(
            "query_row", pa.array(np.arange(len(roster), dtype=np.int32))
        ),
    )
    table = db.execute(
        """WITH grouped AS (
        SELECT q.query_row,t.article_id,count(*) AS n,max(t.t_dat) AS last_date
        FROM transactions t JOIN gpu_anchor_queries q USING(customer_id)
        WHERE t.t_dat < ? GROUP BY q.query_row,t.article_id),
        limited AS (SELECT * FROM grouped QUALIFY row_number() OVER
          (PARTITION BY query_row ORDER BY last_date DESC,article_id) <= ?)
        SELECT l.query_row,m.item_idx,l.n,date_diff('day',l.last_date,?) AS age
        FROM limited l JOIN article_id_map m USING(article_id)
        ORDER BY l.query_row,l.last_date DESC,l.article_id""",
        [anchor, MAX_HISTORY_ITEMS, anchor],
    ).to_arrow_table()
    qrows = table["query_row"].to_numpy().astype(np.int32)
    items = table["item_idx"].to_numpy().astype(np.int32)
    counts = table["n"].to_numpy().astype(np.float64)
    ages = table["age"].to_numpy()
    unique_ages, inverse = np.unique(ages, return_inverse=True)
    # Scalar math.exp matches the prior float64 history-weight calculation exactly.
    decay = np.fromiter(
        (math.exp(-int(age) / 30.0) for age in unique_ages), dtype=np.float64
    )
    weights = counts * decay[inverse]
    indptr = np.concatenate(
        ([0], np.cumsum(np.bincount(qrows, minlength=len(roster))))
    ).astype(np.int64)
    require(
        np.diff(indptr).max(initial=0) <= MAX_HISTORY_ITEMS, "History limit changed."
    )
    history = {"indptr": indptr, "items": items, "weights": weights}
    record = performance_event(
        "anchor_history",
        started,
        anchor=str(anchor),
        queries=len(roster),
        history_rows=len(items),
        array_bytes=sum(v.nbytes for v in history.values()),
    )
    history["seconds"] = record["seconds"]
    return history


def source_item_mapping(db, model_ids):
    mapping = db.execute(
        "SELECT article_id,item_idx FROM article_id_map ORDER BY item_idx"
    ).to_arrow_table()
    require(
        np.array_equal(mapping["item_idx"].to_numpy(), np.arange(len(mapping))),
        "Canonical item mapping is not contiguous.",
    )
    positions = pc.index_in(mapping["article_id"], value_set=pa.array(model_ids))
    return pc.fill_null(positions, -1).to_numpy().astype(np.int32)


def empty_results(n):
    return (
        np.full((n, 100), -1, dtype=np.int32),
        np.zeros((n, 100), dtype=np.float32),
        np.zeros(n, dtype=np.int16),
    )


def store_ranked(target, row, positions, scores, fallback):
    ids, values, lengths = target
    if len(positions) == 0:
        positions, scores = fallback
    # The caller selected Top-100 using original precision; persist/order float32.
    scores = np.asarray(scores, dtype=np.float32)
    positions = np.asarray(positions, dtype=np.int32)
    order = np.lexsort((positions, -scores))
    n = len(order)
    require(0 < n <= 100, "Invalid recommendation count.")
    ids[row, :n], values[row, :n], lengths[row] = positions[order], scores[order], n


def compact_image_preferences(history, start, end, source_to_model, vectors):
    result = np.zeros((end - start, vectors.shape[1]), dtype=np.float32)
    bases = history["indptr"][start:end]
    lengths = np.diff(history["indptr"][start : end + 1])
    # At each history position, update different queries in parallel. Sequential
    # position order and float32 multiply/add match the legacy preference reduction.
    for offset in range(int(lengths.max(initial=0))):
        query_rows = np.flatnonzero(lengths > offset)
        locations = bases[query_rows] + offset
        rows = source_to_model[history["items"][locations]]
        valid = rows >= 0
        query_rows, locations, rows = query_rows[valid], locations[valid], rows[valid]
        if len(rows):
            result[query_rows] += (
                vectors[rows]
                * history["weights"][locations].astype(np.float32)[:, None]
            )
    norms = np.linalg.norm(result, axis=1)
    valid = norms > 1e-12
    result[valid] /= norms[valid, None]
    return result, valid


def compact_index_results(
    index, vectors, valid, model_ids, catalog_positions, fallback
):
    require(model_ids == sorted(set(model_ids)), "Expected sorted unique model IDs.")
    require(index.ntotal == len(model_ids), "FAISS row mapping mismatch.")
    result = empty_results(len(vectors))
    active = np.flatnonzero(valid)
    k = min(100, len(model_ids))
    require(k > 0, "Empty vector catalog.")
    for start in range(0, len(active), QUERY_BATCH_SIZE):
        rows = active[start : start + QUERY_BATCH_SIZE]
        queries = np.ascontiguousarray(vectors[rows], dtype=np.float32)
        scores, positions = index.search(queries, min(k + 1, len(model_ids)))
        # The vectorized common path preserves float32 scores and stable ID ties.
        order = np.lexsort((positions, -scores), axis=1)
        positions = np.take_along_axis(positions, order, axis=1)
        scores = np.take_along_axis(scores, order, axis=1)
        result[0][rows, :k] = catalog_positions[positions[:, :k]]
        result[1][rows, :k] = scores[:, :k]
        result[2][rows] = k
        if scores.shape[1] > k:
            ties = np.flatnonzero(scores[:, k] == scores[:, k - 1])
            for offset in ties:
                threshold = float(
                    np.nextafter(scores[offset, k - 1], np.float32(-np.inf))
                )
                _, tied_scores, tied_positions = index.range_search(
                    queries[offset : offset + 1], threshold
                )
                selected = np.lexsort((tied_positions, -tied_scores))[:k]
                store_ranked(
                    result,
                    int(rows[offset]),
                    catalog_positions[tied_positions[selected]],
                    tied_scores[selected],
                    fallback,
                )
    for row in np.flatnonzero(~valid):
        store_ranked(result, int(row), [], [], fallback)
    return result, int((~valid).sum())


def candidate_arrow(roster, result, catalog, channel):
    ids, scores, lengths = result
    require(len(roster) == len(lengths), "Retriever omitted a query.")
    require(np.all((lengths > 0) & (lengths <= 100)), "Invalid candidate lengths.")
    mask = np.arange(100)[None, :] < lengths[:, None]
    item_rows = ids[mask]
    require(
        np.all((item_rows >= 0) & (item_rows < len(catalog))), "Ineligible item row."
    )
    query_rows = pa.array(np.repeat(np.arange(len(roster), dtype=np.int32), lengths))
    n = len(item_rows)
    zeros = pa.array(np.zeros(n, dtype=np.int8))
    return pa.Table.from_arrays(
        [
            pc.take(roster["query_id"], query_rows),
            pc.take(roster["customer_id"], query_rows),
            pc.take(roster["anchor_date"], query_rows),
            pc.take(catalog, pa.array(item_rows)),
            pc.take(pa.array([channel]), zeros),
            pa.array(scores[mask], type=pa.float32()),
            pa.array(
                np.broadcast_to(np.arange(1, 101, dtype=np.int32), mask.shape)[mask]
            ),
            pc.take(pa.array([ARTIFACT_VERSION]), zeros),
        ],
        schema=CANDIDATE_SCHEMA,
    )


class CandidateWriter:
    def __init__(self, local, attempt):
        self.local = Path(local)
        self.attempt = attempt
        self.writer = None
        self.path = None
        self.files = []
        self.seconds = 0.0
        self.rows = 0

    def write(self, table):
        started = time.perf_counter()
        require(
            table.schema.remove_metadata().equals(CANDIDATE_SCHEMA),
            "Output schema mismatch.",
        )
        if self.writer is None:
            self.path = (
                self.local / f"shard-{self.attempt}-{len(self.files):05d}.parquet"
            )
            self.writer = pq.ParquetWriter(
                self.path, CANDIDATE_SCHEMA, compression="zstd", use_dictionary=True
            )
            self.files.append(self.path)
        self.writer.write_table(table, row_group_size=131072)
        self.rows += len(table)
        if self.path.stat().st_size >= GPU_RESOURCES["target_shard_bytes"]:
            self.writer.close()
            self.writer = None
        self.seconds += time.perf_counter() - started

    def close(self):
        if self.writer is not None:
            started = time.perf_counter()
            self.writer.close()
            self.writer = None
            self.seconds += time.perf_counter() - started


def validate_candidate_files(db, files, roster, catalog, channel):
    require(bool(files), "No candidate shards produced.")
    total = 0
    for path in files:
        file = pq.ParquetFile(path)
        require(
            file.schema_arrow.remove_metadata().equals(CANDIDATE_SCHEMA),
            "Candidate schema mismatch.",
        )
        total += file.metadata.num_rows
    db.register("gpu_validation_queries", roster)
    db.register("gpu_validation_catalog", pa.table({"article_id": catalog}))
    paths = "[" + ",".join(sql_string(p) for p in files) + "]"
    db.execute(
        f"CREATE OR REPLACE VIEW gpu_validate AS SELECT * FROM read_parquet({paths})"
    )
    bad = db.execute(
        """SELECT count(*) FROM gpu_validate c
        LEFT JOIN gpu_validation_queries q USING(query_id)
        WHERE q.query_id IS NULL OR c.customer_id IS DISTINCT FROM q.customer_id
        OR c.anchor_date IS DISTINCT FROM q.anchor_date
        OR c.channel IS DISTINCT FROM ? OR c.retriever_version IS DISTINCT FROM ?
        OR c.raw_score IS NULL OR NOT isfinite(c.raw_score)
        OR c.source_rank IS NULL OR c.article_id IS NULL
        OR NOT EXISTS (SELECT 1 FROM gpu_validation_catalog a
        WHERE a.article_id=c.article_id)""",
        [channel, ARTIFACT_VERSION],
    ).fetchone()[0]
    require(bad == 0, "Candidate IDs, finite scores or query metadata invalid.")
    invalid_groups = db.execute("""SELECT count(*) FROM (
        SELECT query_id FROM gpu_validate GROUP BY query_id
        HAVING count(*)>100 OR count(DISTINCT article_id)<>count(*)
        OR min(source_rank)<>1 OR max(source_rank)<>count(*)
        OR count(DISTINCT source_rank)<>count(*))""").fetchone()[0]
    require(invalid_groups == 0, "Candidate uniqueness/rank contract failed.")
    queries = db.execute(
        "SELECT count(DISTINCT query_id) FROM gpu_validate"
    ).fetchone()[0]
    require(queries == len(roster), "Eligible query coverage changed.")
    unstable = db.execute("""SELECT count(*) FROM (
        SELECT source_rank,row_number() OVER (PARTITION BY query_id
        ORDER BY raw_score DESC,article_id) AS expected FROM gpu_validate)
        WHERE source_rank<>expected""").fetchone()[0]
    require(unstable == 0, "Unstable source score/rank ordering.")
    return total


def pending_gpu_anchor(
    db, roster, catalog, channel, info, manifest, protocol, parameters
):
    stage = candidate_stage(info["split"], channel, info["anchor_date"])
    inputs = {
        "model_manifest": identity(manifest),
        "protocol_manifest": identity(protocol),
    }
    return (
        candidate_reuse(
            db,
            stage,
            parameters,
            inputs,
            identity(catalog),
            info["anchor_date"],
            roster,
        )
        is None
    )

## Frozen learned and text retrieval
Two-Tower checkpoints publish after each completed epoch and resume without consuming later dates. Mini-batches use CUDA mixed precision. Query preferences may advance to each anchor; weights and indexes stay frozen. Deterministic seeds are recorded; exact GPU reproducibility can still depend on hardware and library versions. Missing descriptions retain labeled metadata fields. Missing query history falls back to frozen popularity.

In [ ]:
HISTORY_LENGTH = 32
TOWER_DIM = 64
TOWER_BATCH = 512
TOWER_NEGATIVES = 32
TEXT_MODEL = "sentence-transformers/all-MiniLM-L6-v2"


def make_tower(n_items, categories):
    import torch
    from torch import nn
    from torch.nn import functional as functional

    class TwoTower(nn.Module):
        def __init__(self):
            super().__init__()
            self.item_id = nn.Embedding(n_items + 1, 64, padding_idx=n_items)
            self.categories = nn.ModuleList(
                [
                    nn.Embedding(int(categories[:, j].max()) + 1, 16)
                    for j in range(categories.shape[1])
                ]
            )
            self.register_buffer(
                "category_ids", torch.as_tensor(categories, dtype=torch.long)
            )
            self.item_net = nn.Sequential(
                nn.Linear(64 + 16 * categories.shape[1], 64),
                nn.ReLU(),
                nn.Linear(64, TOWER_DIM),
            )
            self.user_net = nn.Sequential(
                nn.Linear(TOWER_DIM, TOWER_DIM),
                nn.ReLU(),
                nn.Linear(TOWER_DIM, TOWER_DIM),
            )

        def items(self, item_ids):
            blocks = [self.item_id(item_ids)]
            for j, embedding in enumerate(self.categories):
                blocks.append(embedding(self.category_ids[item_ids, j]))
            return functional.normalize(
                self.item_net(torch.cat(blocks, dim=-1)), dim=-1
            )

        def users(self, history_ids):
            mask = (history_ids != n_items).unsqueeze(-1)
            encoded = self.items(history_ids)
            pooled = (encoded * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            return functional.normalize(self.user_net(pooled), dim=-1)

    return TwoTower()


def tower_training_batches(db, item_to_row, n_items):
    from collections import deque

    db.register(
        "training_item_rows",
        pa.table(
            {
                "article_id": list(item_to_row),
                "model_row": pa.array(list(item_to_row.values()), type=pa.int32()),
            }
        ),
    )
    reader = db.execute("""SELECT dense_rank() OVER (ORDER BY customer_id) AS user_row,
        epoch(t_dat)::BIGINT AS day,m.model_row FROM transactions t
        LEFT JOIN training_item_rows m USING(article_id)
        WHERE t_dat < DATE '2020-07-28'
        ORDER BY customer_id,t_dat,article_id""").to_arrow_reader(32768)
    current_user = current_day = None
    history, day_items = deque(maxlen=HISTORY_LENGTH), []
    histories = np.full((TOWER_BATCH, HISTORY_LENGTH), n_items, dtype=np.int32)
    targets = np.empty(TOWER_BATCH, dtype=np.int32)
    filled = 0

    def examples():
        if history:
            padded = np.full(HISTORY_LENGTH, n_items, dtype=np.int32)
            padded[-len(history) :] = history
            for target in day_items:
                yield padded, target

    for batch in reader:
        users = batch.column(0).to_numpy()
        days = batch.column(1).to_numpy()
        items = pc.fill_null(batch.column(2), -1).to_numpy()
        for user, day, item in zip(users, days, items, strict=True):
            if current_user is not None and (
                user != current_user or day != current_day
            ):
                for h, target in examples():
                    histories[filled], targets[filled] = h, target
                    filled += 1
                    if filled == TOWER_BATCH:
                        yield histories.copy(), targets.copy()
                        filled = 0
                for previous_item in day_items:
                    if previous_item in history:
                        history.remove(previous_item)
                    history.append(previous_item)
                day_items = []
                if user != current_user:
                    history.clear()
            current_user, current_day = user, day
            if item >= 0:
                day_items.append(int(item))
    for h, target in examples():
        histories[filled], targets[filled] = h, target
        filled += 1
        if filled == TOWER_BATCH:
            yield histories.copy(), targets.copy()
            filled = 0
    if filled:
        yield histories[:filled].copy(), targets[:filled].copy()


def training_examples(db, catalog, inputs):
    config = {
        "inputs": inputs,
        "catalog_id": identity(catalog),
        "mode": MODE,
        "cutoff": str(CUTOFF),
        "history_length": HISTORY_LENGTH,
        "batch_size": TOWER_BATCH,
        "order": "customer,date,article; no shuffle; v1",
    }
    root = WORK_ROOT / "tower_examples" / RUN_NAMESPACE / MODE / identity(config)
    root.mkdir(parents=True, exist_ok=True)
    marker = root / "_COMPLETE.json"
    if marker.exists():
        saved = json.loads(marker.read_text())
        require(saved["config"] == config, "Training examples configuration mismatch.")
        path = root / saved["file"]
        require(
            sha256_file(path) == saved["sha256"], "Training examples checksum mismatch."
        )
        require(
            pq.ParquetFile(path).metadata.num_rows == saved["rows"],
            "Example count mismatch.",
        )
        return path
    started = time.perf_counter()
    max_rows = db.execute(
        "SELECT count(*) FROM transactions WHERE t_dat < DATE '2020-07-28'"
    ).fetchone()[0]
    disk_guard(root, max_rows * (HISTORY_LENGTH + 1) * 4, "Reusable training examples")
    path = root / f"examples-{uuid.uuid4().hex}.parquet"
    schema = pa.schema(
        [("history", pa.list_(pa.int32(), HISTORY_LENGTH)), ("target", pa.int32())]
    )
    count = 0
    buffered, buffered_rows = [], 0
    with pq.ParquetWriter(path, schema, compression="zstd") as writer:
        for histories, targets in tower_training_batches(
            db, dict(zip(catalog, range(len(catalog)), strict=True)), len(catalog)
        ):
            table = pa.Table.from_arrays(
                [
                    pa.FixedSizeListArray.from_arrays(
                        pa.array(histories.ravel()), HISTORY_LENGTH
                    ),
                    pa.array(targets),
                ],
                schema=schema,
            )
            buffered.append(table)
            buffered_rows += len(table)
            count += len(table)
            if buffered_rows >= 32768:
                writer.write_table(pa.concat_tables(buffered), row_group_size=32768)
                buffered, buffered_rows = [], 0
        if buffered:
            writer.write_table(pa.concat_tables(buffered), row_group_size=32768)
    require(count > 0, "No time-valid Two-Tower examples.")
    require(
        pq.ParquetFile(path).metadata.num_rows == count,
        "Training example write mismatch.",
    )
    atomic_json(
        marker,
        {
            "config": config,
            "file": path.name,
            "rows": count,
            "sha256": sha256_file(path),
        },
    )
    performance_event(
        "tower_training_examples", started, examples=count, bytes=path.stat().st_size
    )
    return path


def cached_training_batches(path):
    for batch in pq.ParquetFile(path).iter_batches(batch_size=TOWER_BATCH):
        histories = batch.column(0).flatten().to_numpy().reshape(-1, HISTORY_LENGTH)
        targets = batch.column(1).to_numpy()
        yield histories.astype(np.int64), targets.astype(np.int64)


def atomic_json(path, value):
    path = Path(path)
    partial = path.with_name(path.name + "." + uuid.uuid4().hex + ".partial")
    write_json(partial, value)
    os.replace(partial, path)  # Local checkpoint metadata only, never a Drive stage.


def validate_tower_checkpoint(checkpoint, epoch, parameters, inputs):
    require(
        {"model", "optimizer", "scaler", "epoch"} <= checkpoint.keys(),
        "Incomplete Two-Tower checkpoint payload.",
    )
    require(checkpoint["epoch"] == epoch, "Checkpoint epoch disagrees with stage.")
    # Legacy checkpoints use the verified outer manifest as their config contract.
    if "parameters" in checkpoint or "inputs" in checkpoint:
        require(
            checkpoint.get("parameters") == parameters
            and checkpoint.get("inputs") == inputs,
            "Checkpoint payload configuration mismatch.",
        )


def compact_tower_history(history, start, end, source_to_model, n_items):
    encoded = np.full((end - start, HISTORY_LENGTH), n_items, dtype=np.int64)
    lengths = np.zeros(end - start, dtype=np.int32)
    bases = history["indptr"][start:end]
    available = np.diff(history["indptr"][start : end + 1])
    # Keep the original recent-first ordering, filter to C0, then take 32.
    for offset in range(int(available.max(initial=0))):
        rows = np.flatnonzero((available > offset) & (lengths < HISTORY_LENGTH))
        items = source_to_model[history["items"][bases[rows] + offset]]
        keep = items >= 0
        rows, items = rows[keep], items[keep]
        encoded[rows, lengths[rows]] = items
        lengths[rows] += 1
    columns = np.arange(HISTORY_LENGTH)[None, :]
    source_columns = columns - (HISTORY_LENGTH - lengths[:, None])
    result = np.take_along_axis(encoded, np.maximum(source_columns, 0), axis=1)
    result[source_columns < 0] = n_items
    return result, lengths > 0


def fit_two_tower(db, catalog, processed_manifest):
    import faiss

    params = {
        "cutoff": str(CUTOFF),
        "epochs": 1 if MODE == "smoke" else 3,
        "history_length": HISTORY_LENGTH,
        "batch_size": TOWER_BATCH,
        "negative_count": TOWER_NEGATIVES,
        "dimension": TOWER_DIM,
        "temperature": 0.1,
        "learning_rate": 0.001,
        "seed": SEED,
    }
    inputs = {"processed_manifest": identity(processed_manifest)}
    reused = reusable_gpu_model("two_tower", params, inputs, catalog, TOWER_DIM)
    if reused is not None:
        return reused
    import torch
    from torch.nn import functional as functional

    require(
        torch.cuda.is_available(),
        ("Two-Tower requires Colab CUDA; preserve its native PyTorch installation."),
    )
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)
    db.register("tower_catalog", pa.table({"article_id": catalog}))
    metadata = db.execute(
        "SELECT article_id,coalesce(product_type_name,'[MISSING]'),\n"
        "        "
        "coalesce(product_group_name,'[MISSING]'),coalesce(colour_group_name,'[MISSING]')"
        "\n"
        "        FROM articles JOIN tower_catalog USING(article_id) "
        "ORDER BY article_id"
    ).fetchall()
    require(
        [r[0] for r in metadata] == catalog, "Tower metadata/item alignment mismatch."
    )
    categories = np.zeros((len(catalog) + 1, 3), dtype=np.int64)
    for j in range(3):
        mapping = {
            v: i + 1 for i, v in enumerate(sorted({row[j + 1] for row in metadata}))
        }
        categories[:-1, j] = [mapping[row[j + 1]] for row in metadata]
    model = make_tower(len(catalog), categories).cuda()
    optimizer = torch.optim.AdamW(model.parameters(), lr=params["learning_rate"])
    scaler = torch.amp.GradScaler("cuda")
    completed = 0
    # Resume completed epochs; restart an interrupted epoch deterministically.
    for epoch in range(params["epochs"], 0, -1):
        stage = f"retrievers/two_tower/checkpoints/epoch-{epoch}"
        if (stage_path(stage, DRIVE_ROOT) / "_COMPLETE.json").exists():
            path, manifest = load_stage(stage, identity(catalog))
            require(
                manifest["input_fingerprints"] == inputs
                and manifest["model_parameters"] == params,
                "Two-Tower checkpoint inputs/configuration mismatch.",
            )
            checkpoint = torch.load(
                path / "checkpoint.pt", map_location="cuda", weights_only=False
            )
            validate_tower_checkpoint(checkpoint, epoch, params, inputs)
            model.load_state_dict(checkpoint["model"])
            optimizer.load_state_dict(checkpoint["optimizer"])
            scaler.load_state_dict(checkpoint["scaler"])
            completed = epoch
            break
    example_path = (
        training_examples(db, catalog, inputs) if completed < params["epochs"] else None
    )
    require(len(catalog) > 1, "Two-Tower needs at least two eligible items.")
    for epoch in range(completed, params["epochs"]):
        model.train()
        generator = torch.Generator(device="cuda").manual_seed(SEED + epoch)
        epoch_started = time.perf_counter()
        total = 0
        loss_sum = torch.zeros((), device="cuda", dtype=torch.float64)
        for histories, targets in cached_training_batches(example_path):
            h = torch.from_numpy(histories).pin_memory().to("cuda", non_blocking=True)
            positive = (
                torch.from_numpy(targets).pin_memory().to("cuda", non_blocking=True)
            )
            negatives = torch.randint(
                len(catalog) - 1,
                (len(targets), TOWER_NEGATIVES),
                device="cuda",
                generator=generator,
            )
            negatives += (negatives >= positive[:, None]).long()
            options = torch.cat((positive[:, None], negatives), dim=1)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                user_vectors = model.users(h)
                item_vectors = model.items(options)
                logits = (item_vectors * user_vectors[:, None, :]).sum(dim=-1) / params[
                    "temperature"
                ]
                loss = functional.cross_entropy(
                    logits.float(),
                    torch.zeros(len(targets), device="cuda", dtype=torch.long),
                )
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer)
            scaler.update()
            loss_sum += loss.detach().double() * len(targets)
            total += len(targets)
        require(total > 0, "No time-valid Two-Tower training examples.")
        mean_loss = float(loss_sum) / total  # One host synchronization per epoch.
        performance_event(
            "tower_epoch",
            epoch_started,
            epoch=epoch + 1,
            examples=total,
            mean_loss=mean_loss,
        )
        checkpoint_started = time.perf_counter()
        stage = f"retrievers/two_tower/checkpoints/epoch-{epoch + 1}"
        local = new_stage(stage)
        torch.save(
            {
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scaler": scaler.state_dict(),
                "epoch": epoch + 1,
                "parameters": params,
                "inputs": inputs,
            },
            local / "checkpoint.pt",
        )
        publish(stage, local, params, inputs, identity(catalog), total)
        performance_event(
            "tower_checkpoint_save_publish", checkpoint_started, epoch=epoch + 1
        )
    local = new_stage("retrievers/two_tower")
    np.save(local / "categories.npy", categories)
    torch.save(model.state_dict(), local / "model.pt")
    pq.write_table(
        pa.table({"article_id": catalog, "embedding_row": range(len(catalog))}),
        local / "items.parquet",
    )
    vectors = np.lib.format.open_memmap(
        local / "vectors.npy",
        mode="w+",
        dtype=np.float32,
        shape=(len(catalog), TOWER_DIM),
    )
    model.eval()
    with torch.inference_mode():
        for start in range(0, len(catalog), 1024):
            positions = torch.arange(
                start, min(start + 1024, len(catalog)), device="cuda"
            )
            vectors[start : start + len(positions)] = (
                model.items(positions).float().cpu().numpy()
            )
    vectors.flush()
    index = flat_index(vectors)
    faiss.write_index(index, str(local / "index.faiss"))
    reloaded = make_tower(len(catalog), categories).cuda().eval()
    reloaded.load_state_dict(torch.load(local / "model.pt", weights_only=True))
    with torch.inference_mode():
        sample = torch.arange(min(32, len(catalog)), device="cuda")
        require(
            torch.allclose(model.items(sample), reloaded.items(sample)),
            "Tower reload parity failed.",
        )
    verify_vector_index(local, catalog, TOWER_DIM)
    del model, reloaded, optimizer, vectors, index
    torch.cuda.empty_cache()
    manifest = publish(
        "retrievers/two_tower", local, params, inputs, identity(catalog), len(catalog)
    )
    return local, manifest


TEXT_FIELDS = [
    "prod_name",
    "product_type_name",
    "product_group_name",
    "colour_group_name",
    "detail_desc",
]


def text_parameters(revision):
    require(
        isinstance(revision, str)
        and len(revision) == 40
        and all(c in "0123456789abcdef" for c in revision.lower()),
        "Expected immutable model revision.",
    )
    return {
        "model": TEXT_MODEL,
        "revision": revision,
        "batch_size": 128,
        "text_fields": TEXT_FIELDS,
        "normalize": True,
    }


def format_text(row):
    return "; ".join(
        f"{field}: {value if value and str(value).strip() else '[missing]'}"
        for field, value in zip(TEXT_FIELDS, row, strict=True)
    )


def validate_vectors(vectors, dimension):
    require(
        vectors.ndim == 2
        and vectors.shape[1] == dimension
        and vectors.dtype == np.float32,
        "Embedding shape/dtype mismatch.",
    )
    for start in range(0, len(vectors), 4096):
        block = vectors[start : start + 4096]
        require(
            np.isfinite(block).all()
            and np.allclose(np.linalg.norm(block, axis=1), 1.0, atol=0.002),
            "Invalid normalized embeddings.",
        )


def text_encoding_workspace(catalog, inputs, resolve_revision):
    root = WORK_ROOT / "text_encoding" / RUN_NAMESPACE / MODE
    root.mkdir(parents=True, exist_ok=True)
    config_file = root / "encoding_config.json"
    expected = {
        "model": TEXT_MODEL,
        "inputs": inputs,
        "catalog_id": identity(catalog),
        "article_order": identity(catalog),
        "mode": MODE,
        "text_fields": TEXT_FIELDS,
        "format": "labeled fields; semicolon-space; [missing]; v1",
        "batch_size": 128,
        "normalize": True,
        "chunk_rows": 1024,
        "dimension": 384,
    }
    if config_file.exists():
        config = json.loads(config_file.read_text())
        require(
            {k: v for k, v in config.items() if k != "revision"} == expected,
            "Text encoding workspace inputs/catalog/preprocessing mismatch.",
        )
        text_parameters(config["revision"])
        return root, config
    # Recover the old revision before any final-stage initialization can rename it.
    legacy = stage_path("retrievers/text", WORK_ROOT) / "encoding_config.json"
    if legacy.exists():
        old = json.loads(legacy.read_text())
        require(
            old["model"] == TEXT_MODEL and old["inputs"] == inputs,
            "Legacy text config mismatch.",
        )
        revision = old["revision"]
    else:
        revision = resolve_revision()
    text_parameters(revision)
    config = {**expected, "revision": revision}
    atomic_json(config_file, config)
    return root, config


def encode_text_batches(root, config, texts, encode):
    require(
        config["article_order"] == config["catalog_id"],
        "Text ordering identity mismatch.",
    )
    started = time.perf_counter()
    paths, encoded, reused = [], 0, 0
    for start in range(0, len(texts), config["chunk_rows"]):
        stop = min(start + config["chunk_rows"], len(texts))
        marker = root / f"batch-{start:08d}.json"
        expected = {
            "config_id": identity(config),
            "start": start,
            "stop": stop,
            "text_id": identity(texts[start:stop]),
        }
        if marker.exists():
            metadata = json.loads(marker.read_text())
            require(
                all(metadata.get(k) == v for k, v in expected.items()),
                "Text batch identity mismatch.",
            )
            filename = metadata["file"]
            require(Path(filename).name == filename, "Unsafe text checkpoint filename.")
            path = root / filename
            require(
                path.is_file() and sha256_file(path) == metadata["sha256"],
                "Text batch checksum mismatch.",
            )
            vectors = np.load(path, mmap_mode="r", allow_pickle=False)
            reused += stop - start
        else:
            vectors = np.asarray(encode(texts[start:stop]), dtype=np.float32)
            validate_vectors(vectors, config["dimension"])
            require(len(vectors) == stop - start, "Text encoder omitted rows.")
            path = root / f"batch-{start:08d}-{uuid.uuid4().hex}.npy"
            np.save(path, vectors)
            # The checksum-bearing marker is committed only after the batch is complete.
            atomic_json(
                marker, {**expected, "file": path.name, "sha256": sha256_file(path)}
            )
            encoded += stop - start
        validate_vectors(vectors, config["dimension"])
        require(len(vectors) == stop - start, "Text checkpoint row mismatch.")
        paths.append(path)
    record = performance_event(
        "text_encoding",
        started,
        encoded_rows=encoded,
        reused_rows=reused,
        encoded_rows_per_second=encoded / max(time.perf_counter() - started, 1e-9),
    )
    record["encoded_rows_per_second"] = encoded / max(record["seconds"], 1e-9)
    return paths


def prepare_text_model(db, catalog, processed_manifest):
    import faiss

    inputs = {"processed_manifest": identity(processed_manifest)}
    marker = stage_path("retrievers/text", DRIVE_ROOT) / "_COMPLETE.json"
    if marker.exists():
        metadata = json.loads(marker.read_text())
        params = text_parameters(metadata["model_parameters"]["revision"])
        return reusable_gpu_model("text", params, inputs, catalog, 384)
    import torch

    require(torch.cuda.is_available(), "Text encoding requires Colab CUDA.")
    from huggingface_hub import HfApi
    from sentence_transformers import SentenceTransformer

    root, config = text_encoding_workspace(
        catalog, inputs, lambda: HfApi().model_info(TEXT_MODEL).sha
    )
    params = text_parameters(config["revision"])
    db.register("text_catalog", pa.table({"article_id": catalog}))
    rows = db.execute("""SELECT a.article_id,a.prod_name,a.product_type_name,
        a.product_group_name,a.colour_group_name,a.detail_desc FROM articles a
        JOIN text_catalog c USING(article_id) ORDER BY a.article_id""").fetchall()
    require([r[0] for r in rows] == catalog, "Text ID alignment mismatch.")
    texts = [format_text(row[1:]) for row in rows]
    del rows
    disk_guard(root, len(catalog) * 384 * 4 * 4, "Text encoding checkpoints and index")
    encoder = None

    def encode(batch):
        nonlocal encoder
        if encoder is None:
            encoder = SentenceTransformer(
                TEXT_MODEL, revision=config["revision"], device="cuda"
            )
            require(
                encoder.get_sentence_embedding_dimension() == 384,
                "Unexpected text dimension.",
            )
        return encoder.encode(
            batch,
            batch_size=128,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False,
        )

    paths = encode_text_batches(root, config, texts, encode)
    del encoder, texts
    torch.cuda.empty_cache()
    local = new_stage("retrievers/text")
    write_json(
        local / "encoding_config.json",
        {"model": TEXT_MODEL, "revision": config["revision"], "inputs": inputs},
    )
    vectors = np.lib.format.open_memmap(
        local / "vectors.npy", mode="w+", dtype=np.float32, shape=(len(catalog), 384)
    )
    offset = 0
    for path in paths:
        chunk = np.load(path, mmap_mode="r", allow_pickle=False)
        vectors[offset : offset + len(chunk)] = chunk
        offset += len(chunk)
    require(offset == len(catalog), "Incomplete text embedding matrix.")
    vectors.flush()
    validate_vectors(vectors, 384)
    pq.write_table(
        pa.table({"article_id": catalog, "embedding_row": range(len(catalog))}),
        local / "items.parquet",
    )
    index = flat_index(vectors)
    faiss.write_index(index, str(local / "index.faiss"))
    verify_vector_index(local, catalog, 384)
    del index, vectors
    manifest = publish(
        "retrievers/text", local, params, inputs, identity(catalog), len(catalog)
    )
    return local, manifest


def verify_vector_index(local, catalog, dimension):
    import faiss

    mapping = pq.read_table(local / "items.parquet")
    require(
        mapping["article_id"].to_pylist() == catalog
        and mapping["embedding_row"].to_pylist() == list(range(len(catalog))),
        "Vector ID alignment mismatch.",
    )
    vectors = np.load(local / "vectors.npy", mmap_mode="r")
    require(len(vectors) == len(catalog), "Vector catalog size mismatch.")
    validate_vectors(vectors, dimension)
    index = faiss.read_index(str(local / "index.faiss"))
    require(
        index.ntotal == len(catalog)
        and index.d == dimension
        and isinstance(index, faiss.IndexFlatIP),
        "Expected exact inner-product index.",
    )
    for start in range(0, len(vectors), 4096):
        count = min(4096, len(vectors) - start)
        require(
            np.array_equal(
                index.reconstruct_n(start, count), vectors[start : start + count]
            ),
            "Saved FAISS row alignment mismatch.",
        )


def emit_gpu_anchor(
    db,
    roster,
    catalog,
    channel,
    info,
    model_manifest,
    protocol_manifest,
    parameters,
    history,
    user_vectors,
    index,
    model_ids,
    fallback,
):
    stage = candidate_stage(info["split"], channel, info["anchor_date"])
    inputs = {
        "model_manifest": identity(model_manifest),
        "protocol_manifest": identity(protocol_manifest),
    }
    local = new_stage(stage)
    disk_guard(local, len(roster) * 100 * 180, "Candidate anchor output estimate")
    attempt = "gpu2-" + uuid.uuid4().hex[:12]
    writer = CandidateWriter(local, attempt)
    vector_seconds = search_seconds = arrow_seconds = 0.0
    fallback_queries = 0
    catalog_arrow = pa.array(catalog)
    positions = np.searchsorted(catalog, model_ids).astype(np.int32)
    try:
        for start in range(0, len(roster), QUERY_BATCH_SIZE):
            end = min(start + QUERY_BATCH_SIZE, len(roster))
            timer = time.perf_counter()
            vectors, valid = user_vectors(start, end)
            vector_seconds += time.perf_counter() - timer
            timer = time.perf_counter()
            result, missing = compact_index_results(
                index, vectors, valid, model_ids, positions, fallback
            )
            search_seconds += time.perf_counter() - timer
            fallback_queries += missing
            timer = time.perf_counter()
            table = candidate_arrow(
                roster.slice(start, end - start), result, catalog_arrow, channel
            )
            arrow_seconds += time.perf_counter() - timer
            writer.write(table)
    finally:
        writer.close()
    timer = time.perf_counter()
    count = validate_candidate_files(db, writer.files, roster, catalog, channel)
    require(count == writer.rows, "Candidate row count mismatch.")
    validation_seconds = time.perf_counter() - timer
    diagnostics = {
        "channel": channel,
        "anchor": str(info["anchor_date"]),
        "queries": len(roster),
        "candidate_rows": count,
        "fallback_queries": fallback_queries,
        "history_seconds": history["seconds"],
        "user_vector_seconds": vector_seconds,
        "faiss_search_seconds": search_seconds,
        "candidate_arrow_seconds": arrow_seconds,
        "parquet_write_seconds": writer.seconds,
        "validation_seconds": validation_seconds,
        "faiss_queries_per_second": len(roster) / max(search_seconds, 1e-9),
        "parquet_files": len(writer.files),
        "parquet_bytes": sum(path.stat().st_size for path in writer.files),
        "process_peak_RSS_GiB": rss_gib(),
        **gpu_memory_stats(),
    }
    write_json(local / f"diagnostics-{attempt}.json", diagnostics)
    timer = time.perf_counter()
    publish(
        stage, local, parameters, inputs, identity(catalog), count, info["anchor_date"]
    )
    diagnostics["publication_seconds"] = time.perf_counter() - timer
    total = (
        vector_seconds
        + search_seconds
        + arrow_seconds
        + writer.seconds
        + validation_seconds
        + diagnostics["publication_seconds"]
    )
    diagnostics["candidate_rows_per_second"] = count / max(total, 1e-9)
    GPU_PERFORMANCE.append(diagnostics)
    print(json.dumps(diagnostics, sort_keys=True))
    return diagnostics


def tower_query_vectors(model, history, start, end, mapping, n_items):
    import torch

    encoded, valid = compact_tower_history(history, start, end, mapping, n_items)
    output = np.empty((end - start, TOWER_DIM), dtype=np.float32)
    # Retain legacy inference microbatch size for bounded VRAM and kernel stability.
    with torch.inference_mode():
        for offset in range(0, len(encoded), 256):
            tensor = torch.from_numpy(encoded[offset : offset + 256]).pin_memory()
            output[offset : offset + len(tensor)] = (
                model.users(tensor.to("cuda", non_blocking=True)).float().cpu().numpy()
            )
    return output, valid


def run_gpu_retrievers():
    from functools import partial

    import faiss
    import torch

    GPU_PERFORMANCE.clear()
    started = time.perf_counter()
    faiss.omp_set_num_threads(THREADS)
    torch.set_num_threads(THREADS)
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    db, catalog, processed_manifest, protocol_manifest = open_context()
    parameters = {c: gpu_candidate_parameters(c) for c in ("two_tower", "text")}
    pop_path, _ = load_stage("retrievers/popularity", identity(catalog))
    fallback = db.execute(
        "SELECT article_id,score FROM read_parquet(?) "
        "ORDER BY score DESC,article_id LIMIT 100",
        [str(pop_path / "popularity.parquet")],
    ).fetchall()
    require(len(fallback) == min(100, len(catalog)), "Incomplete popularity fallback.")
    fallback = (
        np.searchsorted(catalog, [i for i, _ in fallback]).astype(np.int32),
        np.asarray([s for _, s in fallback], dtype=np.float32),
    )
    tower_artifact = fit_two_tower(db, catalog, processed_manifest)
    text_artifact = prepare_text_model(db, catalog, processed_manifest)
    model = tower_index = text_vectors = text_index = mapping = None
    for info in anchor_schedule():
        roster = anchor_roster(db, info["anchor_date"])
        tower_pending = pending_gpu_anchor(
            db,
            roster,
            catalog,
            "two_tower",
            info,
            tower_artifact[1],
            protocol_manifest,
            parameters["two_tower"],
        )
        text_pending = pending_gpu_anchor(
            db,
            roster,
            catalog,
            "text",
            info,
            text_artifact[1],
            protocol_manifest,
            parameters["text"],
        )
        if not (tower_pending or text_pending):
            continue
        history = prepare_anchor_history(db, roster, info["anchor_date"])
        if mapping is None:
            mapping = source_item_mapping(db, catalog)
        if tower_pending:
            if model is None:
                require(torch.cuda.is_available(), "Two-Tower inference requires CUDA.")
                categories = np.load(tower_artifact[0] / "categories.npy")
                model = make_tower(len(catalog), categories).cuda().eval()
                model.load_state_dict(
                    torch.load(tower_artifact[0] / "model.pt", weights_only=True)
                )
                tower_index = faiss.read_index(str(tower_artifact[0] / "index.faiss"))
            compute = partial(
                tower_query_vectors,
                model,
                history,
                mapping=mapping,
                n_items=len(catalog),
            )
            emit_gpu_anchor(
                db,
                roster,
                catalog,
                "two_tower",
                info,
                tower_artifact[1],
                protocol_manifest,
                parameters["two_tower"],
                history,
                compute,
                tower_index,
                catalog,
                fallback,
            )
        if text_pending:
            if text_vectors is None:
                text_vectors = np.load(text_artifact[0] / "vectors.npy", mmap_mode="r")
                if text_vectors.nbytes < GPU_RESOURCES["usable_ram_bytes"] / 10:
                    text_vectors = np.asarray(text_vectors).copy()
                text_index = faiss.read_index(str(text_artifact[0] / "index.faiss"))
            compute = partial(
                compact_image_preferences,
                history,
                source_to_model=mapping,
                vectors=text_vectors,
            )
            emit_gpu_anchor(
                db,
                roster,
                catalog,
                "text",
                info,
                text_artifact[1],
                protocol_manifest,
                parameters["text"],
                history,
                compute,
                text_index,
                catalog,
                fallback,
            )
        history = None
    db.close()
    performance_event("gpu_pipeline_total", started)
    stage = f"retrievers/gpu_performance/{uuid.uuid4().hex[:12]}"
    local = new_stage(stage)
    write_json(local / "performance.json", GPU_PERFORMANCE)
    publish(
        stage,
        local,
        {"implementation": "gpu-execution-2", "resources": GPU_RESOURCES},
        {"processed_manifest": identity(processed_manifest)},
        identity(catalog),
        len(GPU_PERFORMANCE),
    )
    print(json.dumps(performance_summary(), sort_keys=True))


def reusable_gpu_model(channel, params, inputs, catalog, dimension):
    reused = reusable(f"retrievers/{channel}", params, inputs, identity(catalog))
    if reused is None:
        return None
    local, manifest = reused
    required = {"vectors.npy", "items.parquet", "index.faiss"}
    if channel == "two_tower":
        required |= {"categories.npy", "model.pt"}
    require(
        required <= manifest["files"].keys(), "Completed model lacks required files."
    )
    verify_vector_index(local, catalog, dimension)
    if channel == "two_tower":
        categories = np.load(local / "categories.npy", allow_pickle=False)
        require(
            categories.shape == (len(catalog) + 1, 3) and categories.dtype == np.int64,
            "Two-Tower category mapping shape/dtype mismatch.",
        )
    return reused


def performance_summary():
    fields = (
        "queries",
        "candidate_rows",
        "user_vector_seconds",
        "faiss_search_seconds",
        "candidate_arrow_seconds",
        "parquet_write_seconds",
        "publication_seconds",
        "parquet_files",
        "parquet_bytes",
    )
    channels = {}
    for channel in ("two_tower", "text"):
        records = [r for r in GPU_PERFORMANCE if r.get("channel") == channel]
        channels[channel] = {key: sum(r[key] for r in records) for key in fields}
    return {
        "resources": GPU_RESOURCES,
        "newly_generated_channels": channels,
        "reused_candidate_anchors": sum(
            r.get("operation") == "verified_completed_anchor" for r in GPU_PERFORMANCE
        ),
        "model_operations": [
            r
            for r in GPU_PERFORMANCE
            if r.get("operation")
            in ("tower_training_examples", "tower_epoch", "text_encoding")
        ],
        "process_peak_RSS_GiB": rss_gib(),
        **gpu_memory_stats(),
    }

## Execute this stage
This is the entry point. It requires the documented completed prerequisites. No models or datasets are downloaded during local source validation.

### Optimized execution, compatibility and recovery

Run Notebook 02 against the existing verified outputs of 00 and 01; do not rerun them.
After both GPU channels complete, continue with 03 and then 04 under their evaluation
safeguards. This notebook changes neither the frozen C0 nor the 15-anchor protocol.

Full mode detects CPU affinity/container quotas and available RAM. A 12-CPU, 53 GiB
L4 runtime selects up to 8 CPU threads, 13.25 GB DuckDB memory, 2048-query candidate
batches and 192 MiB target compressed shards. At 64+ GiB (including sufficiently
large A100 runtimes), batches grow to 4096 and memory caps at 16 GB. GPU type does
not change the model: training batches remain 512, negatives 32, dimension 64 and
epochs 1 smoke / 3 full. Text encoding remains batch 128, normalized FP32 MiniLM.
Native CUDA PyTorch is preserved. Exact CPU FAISS is used; GPU FAISS is not installed.

Training examples are built once on local disk using typed Arrow columns, then reused
across epochs. Customer/date/article ordering, repeated targets, same-day exclusion,
32-item distinct-history update order, padding and negative-generator seeds are
unchanged. No shuffling is introduced. Pinned transfers and existing FP16 AMP are
used; loss totals synchronize once per epoch. Completed epoch checkpoints retain
weights, optimizer, scaler, epoch, configuration and fingerprints; legacy checkpoints
use their verified outer manifests. An interrupted epoch restarts from the last
completed epoch with its original epoch seed.

Text recovery uses a separate local encoding workspace, preserving the original
revision before initializing final model output. Configuration, catalog/order,
preprocessing, per-batch text fingerprints and checksums are verified. Only completed
validated batches are reused; uncommitted batches are re-encoded. These encoding
checkpoints are **local only**: a deleted Colab runtime loses them. Completed models
and epoch checkpoints are durable Drive artifacts. No partial matrix counts as a model.

Both channels share one strictly pre-anchor history aggregation. The legacy 100-item
history limit precedes model filtering; Two-Tower then takes the first 32 eligible
items, while Text retains count/recency weights. Query-vector inference, exact search,
Arrow formatting, writing and publication have separate timings. Larger exact FAISS
batches can produce tiny FP32 differences across kernels/hardware; boundary ties
still expand through exact range search and use article-ID ordering. No approximation
or model hyperparameter change is introduced.

Candidate filenames and the eight-field schema match reader 03. Legacy manifest
batch-size values remain compatibility descriptors across anchors; actual resource
settings appear under `execution`. Verified completed artifacts are reused unchanged.
New partial candidate attempts use unique filenames, and completion markers list only
validated new files. Existing remote files are not deleted or renamed. Conflicting
partial model/checkpoint files and incompatible completed stages fail for manual
review. Run only one publisher per stage.

Full Colab throughput, CUDA training/resume parity and peak VRAM remain unverified by
local synthetic checks. The final structured summary records actual measurements when
executed; local tests do not establish a Full Mode speedup.


In [ ]:
runtime_start()
metric_smoke_checks()
run_gpu_retrievers()